In [2]:
pip install dspy

  Using cached dspy-3.2.1-py3-none-any.whl.metadata (8.4 kB)
  Using cached litellm-1.86.1-py3-none-any.whl.metadata (33 kB)
  Using cached diskcache-5.6.3-py3-none-any.whl.metadata (20 kB)
  Using cached json_repair-0.59.10-py3-none-any.whl.metadata (19 kB)
  Using cached asyncer-0.0.8-py3-none-any.whl.metadata (6.7 kB)
  Using cached cachetools-7.1.4-py3-none-any.whl.metadata (5.5 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached gepa-0.0.27-py3-none-any.whl.metadata (30 kB)
  Using cached typeguard-4.4.3-py3-none-any.whl.metadata (3.4 kB)
  Using cached fastuuid-0.14.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.1 kB)
  Using cached importlib_metadata-8.9.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached tokenizers-0.23.1-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (9.8 kB)
  Using cached zipp-4.1.0-py3-none-any.whl.metadata (3.6 kB)
  Using cached huggingface_hub-1.16.3-py3-none-any.whl.metadat

In [ ]:
import dspy
import os
from openai import AsyncOpenAI
from agents import set_default_openai_client, set_default_openai_api
import asyncio
from dataclasses import dataclass, field
from pydantic import BaseModel
from agents import Agent, Runner, ModelSettings, handoff, RunContextWrapper, set_tracing_disabled
set_tracing_disabled(True)

client = AsyncOpenAI(
    api_key="OPENAI_API_KEY",
    base_url="https://openrouter.ai/api/v1",
)
set_default_openai_client(client)
set_default_openai_api("chat_completions")

lm = dspy.LM(
    model="openai/gpt-4o",
    api_key="OPENAI_API_KEY",
    api_base="https://openrouter.ai/api/v1",
    max_tokens=512,
)
dspy.configure(lm=lm)

class TransformPrompt(dspy.Signature):
    """Rewrite the prompt to be more specific, structured, and actionable."""
    raw_prompt: str = dspy.InputField()
    transformed_prompt: str = dspy.OutputField()

def dspy_transform(prompt: str) -> str:
    result = dspy.Predict(TransformPrompt)(raw_prompt=prompt)
    return result.transformed_prompt

# ── Shared pipeline context ───────────────────────────────────────────────────
@dataclass
class PipelineContext:
    original_task: str = ""
    transformed_task: str = ""

# ── Handoff payload schema (MasterAgent must fill this) ───────────────────────
class HandoffPayload(BaseModel):
    task: str

# ── Transformation method: replaces MasterAgent output before SubAgent sees it ─
def transform_handoff(ctx: RunContextWrapper[PipelineContext], payload: HandoffPayload):
    original = payload.task
    transformed = dspy_transform(original.strip())

    print(f"[Handoff] Original  → {original}")
    print(f"[Handoff] Replaced  → {transformed}")

    ctx.context.original_task = original
    ctx.context.transformed_task = transformed

# ── Agent 2: Sub Agent (Executor) ─────────────────────────────────────────────
sub_agent = Agent(
    name="SubAgent",
    model="openai/gpt-4o",
    model_settings=ModelSettings(max_tokens=512),
    instructions=(
        "You are an executor agent. "
        "You receive a clear task from the master agent and complete it. "
        "Return only the final result — no explanations."
    ),
)

# ── Agent 1: Master Agent (Orchestrator) ──────────────────────────────────────
master_agent = Agent(
    name="MasterAgent",
    model="openai/gpt-4o",
    model_settings=ModelSettings(max_tokens=512),
    instructions=(
        "You are a master orchestrator. "
        "When the user sends a request, hand it off to SubAgent. "
        "Put the full user request in the 'task' field."
    ),
    handoffs=[
        handoff(
            agent=sub_agent,
            on_handoff=transform_handoff,
            input_type=HandoffPayload,
        )
    ],
)

# ── Run the pipeline ──────────────────────────────────────────────────────────
async def run_pipeline(user_prompt: str) -> str:
    ctx = PipelineContext()
    print(f"\n[User]   {user_prompt[:100]}")
    result = await Runner.run(master_agent, input=user_prompt, context=ctx)
    # print(f"[Output] {result.final_output[:100]}\n")
    return result.final_output
await run_pipeline("Write about china")


[User]   Write about china
[Handoff] Original  → Write about China.
[Handoff] Replaced  → Write a detailed essay on the economic growth of China since 1978, highlighting key reforms, major milestones, and the impact on global trade. Include analyses of government policies, challenges faced, and future outlook.


"China, officially known as the People's Republic of China (PRC), is a country located in East Asia. It is the world's most populous country, with a population of over 1.4 billion people. China is known for its rich history, which dates back thousands of years, and has made significant contributions to human civilization, including the development of paper, gunpowder, the compass, and printing.\n\nThe country is governed by the Communist Party of China, with its capital in Beijing. China has experienced rapid economic growth since the late 20th century, making it the second-largest economy in the world by nominal GDP. It is a major global player in manufacturing, technology, and trade.\n\nChina's culture is diverse and includes influences from Confucianism, Taoism, and Buddhism, among others. The country is known for its traditional art forms, cuisine, and landmarks, including the Great Wall of China, the Forbidden City, and the Terracotta Army.\n\nGeographically, China is vast and div